# 22 — 보강: spoof_type 후처리 + 이미지 기반 Layer 3 캡션 (v2)

> **수정 이력 (v2)**
> - model 직접 로드 (xai_explainer._model 접근 오류 수정)
> - live2 FAIL 대응: 재업로드 + 크롭 마진 조정 셀 추가
> - Print/Mask 보정 기준값: Laplacian 단독 기준으로 단순화 (FFT 차이 너무 작음)
>
> **실행 순서:** Cell 0 → 1 → 2 → **2-b (live2 재업로드, 필요시)** → 3 → 4 → 5 → 6 → 7

## Cell 0 — Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive 마운트 완료')

## Cell 1 — 경로 설정 및 모델 로드

In [ ]:
import os, sys, json, shutil
import numpy as np
import cv2
import tensorflow as tf
from pathlib import Path

BASE       = '/content/drive/MyDrive/face-anti-spoofing'
SRC_DIR    = f'{BASE}/src'
TEST_DIR   = f'{BASE}/data/my_test_images'
REPORT_DIR = f'{BASE}/reports/phase6'
MODEL_PATH = f'{BASE}/models/stage2_webcam_v3.h5'

os.makedirs(REPORT_DIR, exist_ok=True)
sys.path.insert(0, SRC_DIR)

from xai_explainer import explain

# 모델 직접 로드 (xai_explainer.model 접근 불가 대응)
model = tf.keras.models.load_model(MODEL_PATH, compile=False)
print(f'✅ 모델 로드 완료')
print(f'   출력 헤드: {[o.name for o in model.outputs]}')

# 본인 5종 이미지 로드
CATEGORIES = {
    'live'  : ('REAL', '본인 웹캠 Live'),
    'print' : ('FAKE', 'Print Attack'),
    'replay': ('FAKE', 'Replay Attack'),
    'mask'  : ('FAKE', 'Mask Attack'),
    'live2' : ('REAL', 'Live 2번째'),
}

my_images = {}
for cat in CATEGORIES:
    p = f'{TEST_DIR}/my_{cat}.jpg'
    if os.path.exists(p):
        img = cv2.imread(p)
        if img is not None:
            my_images[cat] = (img, p)
            print(f'  ✅ {cat}: {img.shape[1]}×{img.shape[0]}')
    else:
        print(f'  ⚠️ {cat}: 없음 → 21번 노트북 Cell 3 먼저 실행')

print(f'\n총 {len(my_images)}장 로드 완료')

## Cell 2 — (선택) live2 재업로드 + 크롭 재시도

> **live2 FAIL 원인:** Lap=34, FFT=358 → 얼굴이 제대로 크롭되지 않음  
> live2 이미지를 다시 업로드하고 margin을 키워서 얼굴을 더 넓게 잡아요.  
> live2가 이미 PASS라면 이 셀 건너뛰세요.

In [ ]:
from google.colab import files
from IPython.display import display, Image as IPImage

# Haar Cascade 초기화
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

def crop_face(img_bgr, target_size=224, margin_ratio=0.4):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(60,60))
    if len(faces) == 0:
        faces = face_cascade.detectMultiScale(gray, 1.05, 3, minSize=(40,40))
    if len(faces) == 0:
        print('  ⚠️ 얼굴 감지 실패 → center crop')
        h, w = img_bgr.shape[:2]
        s = min(h, w)
        return cv2.resize(img_bgr[(h-s)//2:(h+s)//2, (w-s)//2:(w+s)//2], (target_size, target_size))
    x, y, w, h = max(faces, key=lambda f: f[2]*f[3])
    ih, iw = img_bgr.shape[:2]
    mx, my = int(w*margin_ratio), int(h*margin_ratio)
    x1,y1 = max(0,x-mx), max(0,y-my)
    x2,y2 = min(iw,x+w+mx), min(ih,y+h+my)
    print(f'  얼굴 감지 ✅  박스: ({x1},{y1})-({x2},{y2})')
    return cv2.resize(img_bgr[y1:y2, x1:x2], (target_size, target_size))

print('📤 live2 이미지를 다시 업로드하세요 (정면 얼굴이 잘 보이는 사진 권장)')
uploaded = files.upload()

for fname, fbytes in uploaded.items():
    arr = np.frombuffer(fbytes, dtype=np.uint8)
    img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if img is None:
        print(f'❌ 디코딩 실패: {fname}'); continue

    print(f'원본: {img.shape[1]}×{img.shape[0]}')
    img_cropped = crop_face(img, margin_ratio=0.4)  # 마진 키움

    save_path = f'{TEST_DIR}/my_live2.jpg'
    cv2.imwrite(save_path, img_cropped)
    my_images['live2'] = (img_cropped, save_path)

    # 빠른 확인
    r_check = explain(img_cropped)
    lap = r_check['anchor_stats']['laplacian']
    fft = r_check['anchor_stats']['fft_high']
    print(f'  판정: {r_check["verdict"]} ({r_check["spoof_prob"]:.1%})')
    print(f'  Lap={lap:.1f}, FFT={fft:.1f}')
    if lap < 80:
        print('  ⚠️ Laplacian 여전히 낮음 → 더 선명한 사진으로 교체하거나 margin 줄여보세요')
    else:
        print('  ✅ 수치 정상 범위')
    display(IPImage(save_path, width=160))
    break

## Cell 3 — spoof_type 후처리 보정 함수

> **Print vs Mask 혼동 원인:** softmax argmax만 사용  
> **보정 전략:** 불확실 구간(확률 차 < 0.3)에서 Laplacian 단독 기준으로 보정  
>
> 본인 수치 기반 기준값:
> - Print: Lap=127 / Mask: Lap=188 → **중간값 157**

In [ ]:
SPOOF_KO = {
    0: 'Live (실제 얼굴)',
    1: 'Print Attack (인쇄 공격)',
    2: 'Replay Attack (화면 재촬영)',
    3: '3D Mask (입체 마스크)',
}

def preprocess(img_bgr, size=224):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    return np.expand_dims(
        cv2.resize(img_rgb, (size,size)).astype('float32') / 255.0, 0
    )

# ── 보정 기준값 (본인 수치 기반, 필요시 조정) ──────────────────
LAP_THRESHOLD = 157   # Print(127) / Mask(188) 중간값
UNCERTAIN_GAP = 0.3   # 이 값 미만이면 보정 적용

def anchor_based_type_correction(spoof_type_idx, spoof_type_probs, anchor_stats):
    """
    Print(1) / Mask(3) 혼동 구간에서 Laplacian으로 보정.
    - 두 클래스 확률 차 < UNCERTAIN_GAP 일 때만 보정
    - Lap > LAP_THRESHOLD → Mask / 이하 → Print
    """
    if spoof_type_idx not in (1, 3):
        return spoof_type_idx

    p_print = spoof_type_probs.get(1, 0)
    p_mask  = spoof_type_probs.get(3, 0)

    if abs(p_print - p_mask) >= UNCERTAIN_GAP:
        return spoof_type_idx  # 충분히 확실하면 모델 믿기

    lap = anchor_stats.get('laplacian', 0)
    corrected = 3 if lap > LAP_THRESHOLD else 1

    if corrected != spoof_type_idx:
        print(f'  🔧 후처리 보정: {SPOOF_KO[spoof_type_idx]} → {SPOOF_KO[corrected]}')
        print(f'     Lap={lap:.1f} (기준={LAP_THRESHOLD}), p_print={p_print:.3f}, p_mask={p_mask:.3f}')
    return corrected

print(f'✅ anchor_based_type_correction() 준비')
print(f'   기준값: Laplacian={LAP_THRESHOLD}, 불확실 구간={UNCERTAIN_GAP}')

## Cell 4 — 이미지 기반 Layer 3 캡션 생성 함수

> Grad-CAM 히트맵 위치 + Laplacian/FFT 수치 → 이미지별 구체적 설명 생성

In [ ]:
LIVE_MEAN = {'laplacian': 383.0, 'fft_high': 1134.0}

REGION_MAP = {
    'upper-center': 'forehead region',
    'upper-left':   'forehead-left region',
    'upper-right':  'forehead-right region',
    'mid-center':   'nose and cheek area',
    'mid-left':     'left cheek area',
    'mid-right':    'right cheek area',
    'lower-center': 'mouth and chin area',
    'lower-left':   'lower-left jaw area',
    'lower-right':  'lower-right jaw area',
    'full-face':    'entire face region',
    'none':         'no concentrated region',
}

def detect_heatmap_region(heatmap_raw, threshold=0.5):
    if heatmap_raw is None: return 'unknown'
    hm = cv2.resize(heatmap_raw, (9, 9))
    active = hm >= threshold
    if active.mean() > 0.6: return 'full-face'
    rows = np.where(active.any(axis=1))[0]
    cols = np.where(active.any(axis=0))[0]
    if len(rows) == 0: return 'none'
    v = 'upper' if rows.mean() < 3 else ('lower' if rows.mean() > 6 else 'mid')
    h = 'left'  if cols.mean() < 3 else ('right' if cols.mean() > 6 else 'center')
    return f'{v}-{h}'

def build_image_caption(verdict, spoof_type_idx, anchor_stats, heatmap_raw):
    lap    = anchor_stats.get('laplacian', 0)
    fft    = anchor_stats.get('fft_high', 0)
    region = detect_heatmap_region(heatmap_raw)
    region_desc = REGION_MAP.get(region, region)

    # Laplacian 해석
    if lap < LIVE_MEAN['laplacian'] * 0.5:
        lap_desc = f'very low sharpness (Lap={lap:.0f}, Live avg={LIVE_MEAN["laplacian"]:.0f}) — flat/printed texture'
    elif lap < LIVE_MEAN['laplacian'] * 0.8:
        lap_desc = f'reduced sharpness (Lap={lap:.0f}) — surface artifact'
    elif lap > LIVE_MEAN['laplacian'] * 1.2:
        lap_desc = f'high edge contrast (Lap={lap:.0f}) — mask boundary'
    else:
        lap_desc = f'normal sharpness (Lap={lap:.0f}) — close to live average'

    # FFT 해석
    if fft < LIVE_MEAN['fft_high'] * 0.6:
        fft_desc = f'low high-freq energy (FFT={fft:.0f}) — moire/compression artifact'
    elif fft < LIVE_MEAN['fft_high'] * 0.85:
        fft_desc = f'suppressed high-freq energy (FFT={fft:.0f}) — screen/print attenuation'
    else:
        fft_desc = f'normal high-freq energy (FFT={fft:.0f})'

    type_hints = {
        0: 'No spoofing artifacts detected — classified as live face.',
        1: 'Paper-based attack: flat surface texture and ink dot pattern visible.',
        2: 'Screen replay: digital display interference pattern observed.',
        3: '3D mask: rigid boundary and synthetic skin texture inconsistency present.',
    }

    return (
        f'Model focused on {region_desc}. '
        f'Texture: {lap_desc}, {fft_desc}. '
        f'{type_hints.get(spoof_type_idx, "")}'
    )

print('✅ detect_heatmap_region(), build_image_caption() 준비 완료')

## Cell 5 — explain_v2() 통합 + 5종 검증

In [ ]:
def explain_v2(img_bgr, img_path_str=None, threshold=0.75):
    """
    보강 적용 explain() wrapper:
    - 보강①: spoof_type Laplacian 기반 후처리 보정
    - 보강②: 이미지 특성 기반 Layer 3 캡션
    - 버그 수정: REAL → spoof_type_name 강제 덮어쓰기
    """
    r = explain(img_bgr, img_path=img_path_str, threshold=threshold)

    # 보강①: spoof_type probs 추출 후 보정
    inp   = preprocess(img_bgr)
    preds = model.predict(inp, verbose=0)
    if isinstance(preds, list) and len(preds) >= 2:
        spoof_type_probs = {i: float(p) for i, p in enumerate(preds[1][0])}
    else:
        spoof_type_probs = {}

    corrected_idx = anchor_based_type_correction(
        r['spoof_type_idx'], spoof_type_probs, r['anchor_stats']
    )
    r['spoof_type_idx']  = corrected_idx
    r['spoof_type_name'] = SPOOF_KO.get(corrected_idx, r['spoof_type_name'])

    if r['verdict'] == 'REAL':
        r['spoof_type_name'] = 'Live (실제 얼굴)'

    # 보강②: 이미지 기반 캡션
    r['llava_caption'] = build_image_caption(
        r['verdict'], r['spoof_type_idx'],
        r['anchor_stats'], r['heatmap_raw']
    )

    return r


# ── 5종 검증 ─────────────────────────────────────────────────
print('=' * 65)
print('  본인 5종 이미지 — explain_v2() 검증')
print('=' * 65)

v2_results = {}
pass_count = 0
type_correct = 0

for cat in ['live', 'print', 'replay', 'mask', 'live2']:
    expected, label = CATEGORIES[cat]
    if cat not in my_images:
        print(f'\n[{cat}] ⚠️ 이미지 없음'); continue

    img_bgr, img_path = my_images[cat]
    r = explain_v2(img_bgr, img_path_str=img_path)
    v2_results[cat] = r

    is_correct = (r['verdict'] == expected)
    status = '✅ PASS' if is_correct else '❌ FAIL'
    if is_correct: pass_count += 1

    # spoof_type 정확도 (FAKE 3종)
    type_ok = ''
    if cat in ('print', 'replay', 'mask'):
        expected_type = {'print':'print','replay':'replay','mask':'mask'}[cat]
        type_ok = ' ✅' if expected_type in r['spoof_type_name'].lower() else ' ❌'
        if type_ok == ' ✅': type_correct += 1

    print(f'\n[{cat.upper()}] {label}')
    print(f'  판정    : {r["verdict"]} ({r["spoof_prob"]:.1%})  →  {status}')
    print(f'  유형    : {r["spoof_type_name"]}{type_ok}')
    print(f'  Lap/FFT : {r["anchor_stats"]["laplacian"]:.1f} / {r["anchor_stats"]["fft_high"]:.1f}')
    cap = r.get('llava_caption', '')
    print(f'  캡션    : {cap[:100]}...' if len(cap) > 100 else f'  캡션    : {cap}')

print()
print('=' * 65)
print(f'  판정 결과: {pass_count}/{len(v2_results)} PASS')
print(f'  유형 정확도 (FAKE 3종): {type_correct}/3')
print('=' * 65)

if pass_count < len(v2_results):
    fail_cats = [c for c, r in v2_results.items() if r['verdict'] != CATEGORIES[c][0]]
    print(f'\n⚠️ FAIL: {fail_cats}')
    print('   live/live2 FAIL → Cell 2에서 재업로드 후 재시도')
    print('   print/mask 유형 오류 → Cell 3의 LAP_THRESHOLD 조정')

## Cell 5-b — (선택) LAP_THRESHOLD 튜닝

> Print/Mask 유형 오류가 남아 있을 때 기준값을 직접 조정합니다.  
> 아래 셀에서 본인 수치를 확인하고 중간값을 재계산하세요.

In [ ]:
# 본인 실제 수치 출력
print('[본인 수치 확인]')
for cat in ['print', 'mask']:
    if cat in v2_results:
        r = v2_results[cat]
        lap = r['anchor_stats']['laplacian']
        fft = r['anchor_stats']['fft_high']
        print(f'  {cat:6s}: Lap={lap:.1f}, FFT={fft:.1f}')

# 수치 입력 후 새 기준값 계산
lap_print = v2_results.get('print', {}).get('anchor_stats', {}).get('laplacian', 127)
lap_mask  = v2_results.get('mask',  {}).get('anchor_stats', {}).get('laplacian', 188)
new_threshold = int((lap_print + lap_mask) / 2)

print(f'\n  계산된 최적 LAP_THRESHOLD: {new_threshold}')
print(f'  (기존: {LAP_THRESHOLD})')

# 적용
LAP_THRESHOLD = new_threshold
print(f'\n✅ LAP_THRESHOLD → {LAP_THRESHOLD} 로 업데이트')
print('   Cell 5 재실행하세요')

## Cell 6 — 비교 시각화 (기존 vs v2)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib

font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    prop = fm.FontProperties(fname=font_path)
    matplotlib.rcParams['font.family'] = prop.get_name()
matplotlib.rcParams['axes.unicode_minus'] = False

cat_order = [c for c in ['live','print','replay','mask','live2'] if c in v2_results]
n = len(cat_order)

fig, axes = plt.subplots(n, 4, figsize=(22, 5*n))
fig.suptitle('explain_v2() 보강 결과 비교  |  초록=개선됨',
             fontsize=13, fontweight='bold', y=1.01)
if n == 1: axes = [axes]

for row_i, cat in enumerate(cat_order):
    r_v2 = v2_results[cat]
    img_bgr, img_path = my_images[cat]
    r_old = explain(img_bgr, img_path=img_path)
    expected, label = CATEGORIES[cat]
    is_correct = (r_v2['verdict'] == expected)

    ax = axes[row_i]

    # 원본
    ax[0].imshow(cv2.cvtColor(cv2.resize(img_bgr,(224,224)), cv2.COLOR_BGR2RGB))
    ax[0].set_title(
        f'{label}\n{"✅" if is_correct else "❌"} {r_v2["verdict"]} ({r_v2["spoof_prob"]:.0%})',
        fontsize=9, color='green' if is_correct else 'red')
    ax[0].axis('off')

    # Grad-CAM
    ax[1].imshow(r_v2['heatmap_overlay'])
    region = detect_heatmap_region(r_v2['heatmap_raw'])
    ax[1].set_title(f'Grad-CAM\n활성: {region}', fontsize=9)
    ax[1].axis('off')

    # 기존 캡션
    ax[2].axis('off')
    old_type = r_old.get('spoof_type_name', '?')
    old_cap  = r_old.get('llava_caption', '') or '[없음]'
    type_changed = (old_type != r_v2['spoof_type_name'])
    ax[2].text(0.04, 0.97,
               f'[기존]\n유형: {old_type}\n\n캡션:\n{old_cap[:160]}',
               transform=ax[2].transAxes, fontsize=7.5, verticalalignment='top',
               bbox=dict(boxstyle='round',
                         facecolor='lightyellow' if type_changed else 'whitesmoke',
                         alpha=0.85))
    ax[2].set_title('기존 explain()', fontsize=9)

    # v2
    ax[3].axis('off')
    new_type = r_v2['spoof_type_name']
    new_cap  = r_v2.get('llava_caption', '')
    ax[3].text(0.04, 0.97,
               f'[v2 보강]\n유형: {new_type}\n\n캡션:\n{new_cap[:160]}',
               transform=ax[3].transAxes, fontsize=7.5, verticalalignment='top',
               bbox=dict(boxstyle='round',
                         facecolor='lightgreen' if type_changed else 'whitesmoke',
                         alpha=0.85))
    ax[3].set_title('explain_v2() ← 개선' if type_changed else 'explain_v2()',
                    fontsize=9, color='green' if type_changed else 'black')

plt.tight_layout()
out = f'{REPORT_DIR}/22_v2_comparison.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ 저장: {out}')

## Cell 7 — xai_explainer_v2.py 저장

> 보강 코드를 기존 xai_explainer.py에 append해서 v2 파일로 저장

In [ ]:
with open(f'{SRC_DIR}/xai_explainer.py', 'r', encoding='utf-8') as f:
    original_code = f.read()

addon = f'''

# ════════════════════════════════════════════════════════
# v2 보강 (22번 노트북)
# ════════════════════════════════════════════════════════

LIVE_MEAN    = {{'laplacian': 383.0, 'fft_high': 1134.0}}
LAP_THRESHOLD = {LAP_THRESHOLD}  # Print/Mask 보정 기준값 (본인 수치 기반)

REGION_MAP = {{
    'upper-center':'forehead region','upper-left':'forehead-left region',
    'upper-right':'forehead-right region','mid-center':'nose and cheek area',
    'mid-left':'left cheek area','mid-right':'right cheek area',
    'lower-center':'mouth and chin area','lower-left':'lower-left jaw area',
    'lower-right':'lower-right jaw area','full-face':'entire face region',
    'none':'no concentrated region',
}}

def detect_heatmap_region(heatmap_raw, threshold=0.5):
    if heatmap_raw is None: return 'unknown'
    hm = cv2.resize(heatmap_raw, (9,9))
    active = hm >= threshold
    if active.mean() > 0.6: return 'full-face'
    rows = np.where(active.any(axis=1))[0]
    cols = np.where(active.any(axis=0))[0]
    if len(rows) == 0: return 'none'
    v = 'upper' if rows.mean()<3 else ('lower' if rows.mean()>6 else 'mid')
    h = 'left' if cols.mean()<3 else ('right' if cols.mean()>6 else 'center')
    return f'{{v}}-{{h}}'

def anchor_based_type_correction(spoof_type_idx, spoof_type_probs, anchor_stats):
    if spoof_type_idx not in (1,3): return spoof_type_idx
    p_print = spoof_type_probs.get(1,0)
    p_mask  = spoof_type_probs.get(3,0)
    if abs(p_print-p_mask) >= 0.3: return spoof_type_idx
    lap = anchor_stats.get('laplacian',0)
    return 3 if lap > LAP_THRESHOLD else 1

def build_image_caption(verdict, spoof_type_idx, anchor_stats, heatmap_raw):
    lap = anchor_stats.get('laplacian',0)
    fft = anchor_stats.get('fft_high',0)
    region_desc = REGION_MAP.get(detect_heatmap_region(heatmap_raw), 'face region')
    lap_desc = (
        f'very low sharpness (Lap={{lap:.0f}})' if lap<LIVE_MEAN['laplacian']*0.5 else
        f'reduced sharpness (Lap={{lap:.0f}})' if lap<LIVE_MEAN['laplacian']*0.8 else
        f'high edge contrast (Lap={{lap:.0f}})' if lap>LIVE_MEAN['laplacian']*1.2 else
        f'normal sharpness (Lap={{lap:.0f}})'
    )
    fft_desc = (
        f'low high-freq energy (FFT={{fft:.0f}})' if fft<LIVE_MEAN['fft_high']*0.6 else
        f'suppressed high-freq energy (FFT={{fft:.0f}})' if fft<LIVE_MEAN['fft_high']*0.85 else
        f'normal high-freq energy (FFT={{fft:.0f}})'
    )
    type_hints = {{
        0:'No spoofing artifacts detected — classified as live face.',
        1:'Paper-based attack: flat surface texture and ink dot pattern visible.',
        2:'Screen replay: digital display interference pattern observed.',
        3:'3D mask: rigid boundary and synthetic skin texture inconsistency present.',
    }}
    return (f'Model focused on {{region_desc}}. '
            f'Texture: {{lap_desc}}, {{fft_desc}}. '
            f'{{type_hints.get(spoof_type_idx, "")}}')

def explain_v2(img_bgr, img_path=None, threshold=0.75):
    import tensorflow as tf
    r = explain(img_bgr, img_path=img_path, threshold=threshold)
    _model = tf.keras.models.load_model(
        \'{BASE}/models/stage2_webcam_v3.h5\', compile=False
    ) if not globals().get(\'_v2_model\') else globals()[\'_v2_model\']
    globals()[\'_v2_model\'] = _model
    inp   = np.expand_dims(cv2.resize(
        cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB),(224,224)
    ).astype(\'float32\')/255.0, 0)
    preds = _model.predict(inp, verbose=0)
    spoof_type_probs = {{i:float(p) for i,p in enumerate(preds[1][0])}} if isinstance(preds,list) and len(preds)>=2 else {{}}
    corrected_idx = anchor_based_type_correction(r[\'spoof_type_idx\'],spoof_type_probs,r[\'anchor_stats\'])
    r[\'spoof_type_idx\']  = corrected_idx
    r[\'spoof_type_name\'] = SPOOF_KO.get(corrected_idx, r[\'spoof_type_name\'])
    if r[\'verdict\'] == \'REAL\': r[\'spoof_type_name\'] = \'Live (실제 얼굴)\'
    r[\'llava_caption\'] = build_image_caption(r[\'verdict\'],r[\'spoof_type_idx\'],r[\'anchor_stats\'],r[\'heatmap_raw\'])
    return r
'''

v2_path = f'{SRC_DIR}/xai_explainer_v2.py'
with open(v2_path, 'w', encoding='utf-8') as f:
    f.write(original_code + addon)

print(f'✅ 저장: {v2_path}')
print(f'   LAP_THRESHOLD={LAP_THRESHOLD} 반영됨')

## Cell 8 — (선택) xai_explainer.py 교체

> **5/5 PASS + spoof_type 개선 확인 후에만 실행**

In [ ]:
orig = f'{SRC_DIR}/xai_explainer.py'
bak  = f'{SRC_DIR}/xai_explainer_backup.py'
v2   = f'{SRC_DIR}/xai_explainer_v2.py'

shutil.copy(orig, bak)
print(f'✅ 백업: {bak}')
shutil.copy(v2, orig)
print(f'✅ 교체 완료: xai_explainer.py → v2')
print('\n→ 20번 노트북 Cell 6 재실행하면 앱에 보강 반영됩니다.')

## ✅ 체크리스트

| 항목 | 확인 |
|------|------|
| Cell 1: 모델 + 5종 이미지 로드 | ⬜ |
| Cell 2: live2 재업로드 (Lap < 80이면 필수) | ⬜ |
| Cell 3: anchor_based_type_correction() 준비 | ⬜ |
| Cell 4: build_image_caption() 준비 | ⬜ |
| Cell 5: 5/5 PASS 확인 | ⬜ |
| Cell 5-b: LAP_THRESHOLD 재계산 (유형 오류 시) | ⬜ |
| Cell 6: 비교 시각화 확인 | ⬜ |
| Cell 7: xai_explainer_v2.py 저장 | ⬜ |
| Cell 8: xai_explainer.py 교체 | ⬜ |
| 20번 Cell 6: Streamlit 앱 재실행 | ⬜ |